# Lab Assignment 7 — Multiobjective Optimization (37 algorithms)
Name: Manavaditya Rajesh

Reg. no: 241070040

 Assigned Deep-dive algorithms **3 (Benson's Method), 15 (NSGA-III), 27 (MODE)**.


## 0. Setup

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.getcwd()); sys.path.insert(0, os.path.join(os.getcwd(), 'experiments'))
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
from experiments.common import *
print('algorithms registered:', len(NAMES))

algorithms registered: 37


## 1. Unit tests (toolkit + one key-rule test per algorithm)

In [ ]:
sys.path.insert(0, os.path.join(os.getcwd(), 'tests'))
import test_all
test_all.main()

PASS  utils: car Pareto-front example 
PASS  utils: Deb constrained-dominance table 
PASS  utils: crowding distance, HV, IGD, spacing 
PASS  01 Weighted Sum: convex w=(.5,.5)->x=5; never the middle of the non-convex ZDT2 front 
PASS  02 eps-Constraint: f2<=eps holds and the point is non-dominated 
PASS  03 Benson: f + l = z, l >= 0, solution non-dominated 
PASS  04 Weight metric: p=inf reaches the ZDT2 middle, p=1 does not 
PASS  05 Value function: result maximises the DM utility among random portfolios 
PASS  06 Goal programming: P11 units goal leaves shortfall 100, profit goal met 
PASS  07 Lexicographic: profit first (2400, tie) then units -> (300,200) 
PASS  08 TOPSIS: dominant alternative has closeness 1, anti-ideal has 0 
PASS  09 VIKOR: dominant alternative has S=R=Q=0 
PASS  10 VEGA: sub-population m is selected on objective m only 
PASS  11 MOGA: rank = 1 + number of dominators 
PASS  12 NSGA: every member of front k has higher dummy fitness than every member of front k+1 
PAS

## 2. Problem sanity checks from the assignment (hand-verifiable numbers)

In [ ]:
chk = [('P1 x=5 -> (4,4)', P.P1(), [[5.]]), ('P2 x=2 -> (50,4)', P.P2(), [[2.]]), ('P5 (b,h)=(0.01,0.05) -> 3.925 kg, 0.016 m', P.P5(), [[0.01, 0.05]]),
       ('P6 (0,0)->(0,50); (5,3)->(136,4)', P.P6(), [[0, 0], [5, 3.]]), ('P4 w=(0,0,1,0) -> risk .09, return .15', P.P4(), [[0, 0, 1, 0.]]),
       ('P14 (13,40)->1.53 kg, 8.99 h', P.P14(), [[13, 40.]]), ('P7 (10,80)->(8,96) after sign flip', P.P7(), [[10, 80.]])]
for name, p, X in chk:
    F, CV = p.evaluate(np.array(X, float)); print(name, '->', np.round(F, 4).tolist(), 'CV', CV.tolist())
M3 = P.P10(3); X = np.random.rand(3, 12); X[:, 2:] = 0.5; print('P10 g=0: sum f^2 =', ((M3.evaluate(X)[0]) ** 2).sum(1).round(6))

P1 x=5 -> (4,4) -> [[4.0, 4.0]] CV [0.0]
P2 x=2 -> (50,4) -> [[50.0, 4.0]] CV [0.0]
P5 (b,h)=(0.01,0.05) -> 3.925 kg, 0.016 m -> [[3.925, 0.016]] CV [0.0]
P6 (0,0)->(0,50); (5,3)->(136,4) -> [[0.0, 50.0], [136.0, 4.0]] CV [0.0, 0.0]
P4 w=(0,0,1,0) -> risk .09, return .15 -> [[0.09, -0.15]] CV [0.0]
P14 (13,40)->1.53 kg, 8.99 h -> [[1.53, -8.9888]] CV [0.0]
P7 (10,80)->(8,96) after sign flip -> [[-8.0, -96.0]] CV [0.0]
P10 g=0: sum f^2 = [1. 1. 1.]


## 3. Task A — every algorithm on its first catalogue problem (seed 1, 10,000 evaluations)
Algorithms 1–4 build a front by sweeping weights/eps/reference points; 5–9 and 36–37 return solutions or rankings (shown in later cells).

In [ ]:
rows = []; fronts = {}
for aid in [1, 2, 3, 4]:
    key = TEST[aid][1] if aid != 3 else 'P7'
    t = time.time(); F = classical_front(aid, key); m = metrics(key, F); m['time'] = time.time() - t; rows.append((aid, key, m)); fronts[(aid, key)] = F
for aid in sorted(POP):
    key = TEST[aid][0]
    if aid == 34: key = 'P10_3'
    X, F, t, nfe = run_pop(aid, key, seed=1, evals=10000 if key != 'P9' else 1500)
    m = metrics(key, F); m['time'] = t; rows.append((aid, key, m)); fronts[(aid, key)] = F
print('%3s %-24s %-6s %7s %8s %8s %5s %7s' % ('#', 'algorithm', 'prob', 'HV', 'IGD', 'Spacing', 'n', 'time s'))
for aid, key, m in rows:
    print('%3d %-24s %-6s %7.3f %8s %8.4f %5d %7.1f' % (aid, NAMES[aid][:24], key, m['HV'], '%.4f' % m['IGD'] if m['IGD'] is not None else 'n/a', m['SP'], m['n'], m['time']))

  # algorithm                prob        HV      IGD  Spacing     n  time s
  1 Weighted Sum             P3       0.210   0.3538   0.0000     2     1.1
  2 eps-Constraint           P3       0.514   0.0389   0.0605    19     2.4
  3 Benson's Method          P7       0.673   0.0246   0.0096    15     2.3
  4 Weight Metric (p-norm, T P3       0.517   0.0180   0.0108    21     2.7
 10 VEGA                     P2       1.112   0.0136   0.0375   100     0.0
 11 MOGA                     P3       0.439   0.0560   0.0203    44     0.3
 12 NSGA                     P5       1.190   0.0132   0.0159    57     0.2
 13 NPGA                     P3       0.035   0.4767   0.0421    11     1.8
 14 NSGA-II                  P3       0.510   0.0196   0.0065   100     0.5
 15 NSGA-III                 P10_3    0.561   0.1204   0.0642    88     0.8
 16 SPEA2                    P4       0.968   0.0044   0.0024   100     5.6
 17 PESA-II                  P3       0.528   0.0092   0.0081   100     1.7
 18 MOEA/D  

In [ ]:
sel = [(a, k) for (a, k) in fronts if k in ('P3', 'P5')][:24]
fig, axs = plt.subplots(4, 6, figsize=(16, 9)); axs = axs.ravel()
for ax, (a, k) in zip(axs, sel):
    plot_front(ax, k, fronts[(a, k)]); ax.set_title('%d %s (%s)' % (a, NAMES[a][:16], k), fontsize=7); ax.set_xlabel(''); ax.set_ylabel('')
for ax in axs[len(sel):]: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Algorithms 5-9: value function, goal programming, lexicographic, TOPSIS, VIKOR
from moo.a08_mcdm import *
print('Value function (P4, three investor profiles):')
for a, x, f in value_function(mkprob('P4'), [(2., 1.), (1., 1.), (1., 2.)]): print('  a =', a, ' weights', np.round(x / x.sum(), 3), ' risk/return', np.round(mkprob('P4').denorm(f) * [1, -1], 4))
print('Goal programming P11 (targets 2400, 600):', goal_programming(np.array([2400., 600.]), np.zeros(2), np.ones(2))[:3])
print('Lexicographic P11 (profit, then units):', lexicographic()[0].round(3))
print('TOPSIS closeness P12:', dict(zip(SUPPLIERS, topsis(P12_DATA, P12_W, P12_BENEFIT)[0].round(3))))
for v in (0.25, 0.5, 0.75): print('VIKOR v=%.2f Q:' % v, dict(zip(SUPPLIERS, vikor(P12_DATA, P12_W, P12_BENEFIT, v)[0].round(3))))
w2 = P12_W.copy(); w2[0] -= .1; w2[1] += .1
print('after shifting 0.1 of weight from cost to quality, TOPSIS order:', [SUPPLIERS[i] for i in np.argsort(-topsis(P12_DATA, w2, P12_BENEFIT)[0])])

Value function (P4, three investor profiles):
  a = [2. 1.]  weights [0.    0.395 0.342 0.263]  risk/return [0.0215 0.125 ]
  a = [1. 1.]  weights [0.    0.453 0.455 0.093]  risk/return [0.0302 0.1318]
  a = [1. 2.]  weights [0.    0.422 0.578 0.   ]  risk/return [0.0401 0.1373]
Goal programming P11 (targets 2400, 600): (array([300., 200.]), array([  0., 100.]), array([1.13686838e-13, 0.00000000e+00]))
Lexicographic P11 (profit, then units): [300. 200.]
TOPSIS closeness P12: {'S1': np.float64(0.621), 'S2': np.float64(0.476), 'S3': np.float64(0.63), 'S4': np.float64(0.37), 'S5': np.float64(0.597)}
VIKOR v=0.25 Q: {'S1': np.float64(0.047), 'S2': np.float64(0.238), 'S3': np.float64(0.75), 'S4': np.float64(1.0), 'S5': np.float64(0.464)}
VIKOR v=0.50 Q: {'S1': np.float64(0.093), 'S2': np.float64(0.309), 'S3': np.float64(0.5), 'S4': np.float64(1.0), 'S5': np.float64(0.429)}
VIKOR v=0.75 Q: {'S1': np.float64(0.14), 'S2': np.float64(0.38), 'S3': np.float64(0.25), 'S4': np.float64(1.0), 'S5': n

## 4. Task C — hand traces reproduced by the code
### 4.1 Benson's method (P1, two reference points)

In [ ]:
import math
p1 = mkprob('P1')
for z in [(10., 8.), (2., 20.)]:
    lo = max(1, 3 - math.sqrt(z[0]), 7 - math.sqrt(z[1])); hi = min(10, 3 + math.sqrt(z[0]), 7 + math.sqrt(z[1])); x_hand = min(max(5., lo), hi)
    r = benson(p1, [(np.array(z) - p1.ideal) / (p1.nadir - p1.ideal)])[0]
    print('z =', z, '| hand: x = %.4f, f = (%.4f, %.4f), l = (%.4f, %.4f)' % (x_hand, (x_hand - 3) ** 2, (x_hand - 7) ** 2, z[0] - (x_hand - 3) ** 2, z[1] - (x_hand - 7) ** 2),
          '| code: x = %.4f, f =' % r[1][0], np.round(p1.denorm(r[2]), 4), 'l =', np.round(r[3] * (p1.nadir - p1.ideal), 4))

z = (10.0, 8.0) | hand: x = 5.0000, f = (4.0000, 4.0000), l = (6.0000, 4.0000) | code: x = 5.0000, f = [4. 4.] l = [6. 4.]


z = (2.0, 20.0) | hand: x = 4.4142, f = (2.0000, 6.6863), l = (0.0000, 13.3137) | code: x = 4.4142, f = [2.     6.6863] l = [ 0.     13.3137]


### 4.2 NSGA-III: tiny instance (P1, N = 4, H = 3), two generations

In [ ]:
import runpy
src = open('experiments/run_task_c.py').read(); s0 = src.index('def nsga3_trace'); s1 = src.index('# tiny instance on P1')
ns = {'math': math, 'np': np}; exec(src[s0:s1], ns)
Z = U.das_dennis(2, 3).tolist(); f_ = lambda x: [(x - 3) ** 2, (x - 7) ** 2]
pop = [1.0, 3.5, 6.0, 9.0]
for g, off in enumerate([[1.5, 8.0, 2.0, 5.5], [9.0, 4.5, 4.0, 5.0]]):
    comb = pop + off; Fc = [f_(x) for x in comb]; T = ns['nsga3_trace'](Fc, 4, Z)
    code_sel = sorted(nsga3_select(np.array(Fc), 4, np.array(Z), np.random.default_rng(0)).tolist())
    print('generation', g + 1, '| fronts', T['fronts'], '| intercepts', np.round(T['intercepts'], 3), '| niching', [(n[0], n[3]) for n in T['niching']])
    print('   hand-trace selection', sorted(T['selected']), '| module selection', code_sel, '| MATCH' if sorted(T['selected']) == code_sel else '| MISMATCH')
    pop = [comb[i] for i in T['selected']]

generation 1 | fronts [[1, 2, 7], [5, 6], [3, 4], [0]] | intercepts [ 8.75 11.25] | niching [(0, 6)]
   hand-trace selection [1, 2, 6, 7] | module selection [1, 2, 6, 7] | MATCH
generation 2 | fronts [[0, 1, 2, 5, 6, 7], [3, 4]] | intercepts [ 8.75 11.25] | niching [(0, 0), (1, 5), (2, 7), (3, 1)]
   hand-trace selection [0, 1, 5, 7] | module selection [0, 1, 5, 7] | MATCH


### 4.3 MODE: two iterations, population 4, on P14 (seed 7)

In [ ]:
p14 = mkprob('P14'); a = MODE(p14, pop=4, seed=7, Fscale=0.5, CR=0.9); a.initialize()
rg = np.random.default_rng(7); lb, ub = p14.lb, p14.ub
X = [list(lb + r * (ub - lb)) for r in rg.random((4, 2))]
fobj = lambda x: (0.4 + 0.05 * x[0] + 0.012 * x[1], -(0.2 * x[1] / (0.5 + 0.03 * x[0])))
dom = lambda a_, b_: all(x <= y for x, y in zip(a_, b_)) and any(x < y for x, y in zip(a_, b_))
Fv = [fobj(x) for x in X]
for it in range(2):
    R = [list(rg.choice([j for j in range(4) if j != i], 3, replace=False)) for i in range(4)]; mask = rg.random((4, 2)); jr = rg.integers(2, size=4)
    newX = []
    for i in range(4):
        r1, r2, r3 = R[i]; v = [X[r1][k] + 0.5 * (X[r2][k] - X[r3][k]) for k in range(2)]
        u = [min(max(v[k] if (mask[i][k] < 0.9 or k == jr[i]) else X[i][k], lb[k]), ub[k]) for k in range(2)]
        rep = dom(fobj(u), Fv[i]); print('iter %d target %d r=%s mutant=(%.3f,%.3f) trial=(%.3f,%.3f) f(u)=(%.3f,%.3f) f(x)=(%.3f,%.3f) -> %s' % (it + 1, i, R[i], *v, *u, *fobj(u), *Fv[i], 'replace' if rep else 'keep'))
        newX.append(u if rep else X[i])
    a.iterate(); X = newX; Fv = [fobj(x) for x in X]
    print('   module population identical to hand trace:', np.allclose(a.X, np.array(X)))

iter 1 target 0 r=[np.int64(2), np.int64(3), np.int64(1)] mutant=(12.660,98.578) trial=(13.000,90.000) f(u)=(2.130,-20.225) f(x)=(2.193,-17.587) -> replace
iter 1 target 1 r=[np.int64(3), np.int64(0), np.int64(2)] mutant=(13.671,81.653) trial=(13.671,51.260) f(u)=(1.699,-11.264) f(x)=(1.820,-10.428) -> replace
iter 1 target 2 r=[np.int64(1), np.int64(0), np.int64(3)] mutant=(17.342,53.160) trial=(17.000,53.160) f(u)=(1.888,-10.527) f(x)=(2.114,-18.073) -> keep
iter 1 target 3 r=[np.int64(0), np.int64(2), np.int64(1)] mutant=(14.549,101.069) trial=(14.549,90.000) f(u)=(2.207,-19.221) f(x)=(2.024,-18.203) -> keep
   module population identical to hand trace: True
iter 2 target 0 r=[np.int64(1), np.int64(2), np.int64(3)] mutant=(14.261,52.568) trial=(14.261,52.568) f(u)=(1.744,-11.332) f(x)=(2.130,-20.225) -> keep
iter 2 target 1 r=[np.int64(0), np.int64(3), np.int64(2)] mutant=(12.410,88.692) trial=(13.000,88.692) f(u)=(2.114,-19.931) f(x)=(1.699,-11.264) -> keep
iter 2 target 2 r=[np.in

## 5. Task D (condensed) — constrained P5 and many-objective P10 (M = 5), 3 seeds
The report uses 10 seeds and all algorithms; here three seeds keep the notebook fast.

In [ ]:
def agg(aid, key, seeds=(1, 2, 3)):
    ms = [metrics(key, run_pop(aid, key, seed=s)[1]) for s in seeds]
    return {k: (np.mean([m[k] for m in ms if m[k] is not None]), np.std([m[k] for m in ms if m[k] is not None])) for k in ('HV', 'IGD', 'SP') if ms[0][k] is not None}
print('P5 (constrained)')
for aid in (14, 21, 16, 27, 19):
    r = agg(aid, 'P5'); print('  %-22s HV %.3f+-%.3f  IGD %.4f  Spacing %.4f' % (NAMES[aid], *r['HV'], r['IGD'][0], r['SP'][0]))
print('P10 M=5 (many-objective)')
for aid in (14, 15, 18, 31):
    r = agg(aid, 'P10_5'); print('  %-22s HV %.3f+-%.3f  IGD %.3f' % (NAMES[aid], *r['HV'], r['IGD'][0]))
print('(HypE is omitted here because one run takes about a minute; see report Task D2.)')

P5 (constrained)


  NSGA-II                HV 1.189+-0.002  IGD 0.0093  Spacing 0.0424


  Constrained NSGA-II    HV 1.195+-0.000  IGD 0.0064  Spacing 0.0079


  SPEA2                  HV 1.195+-0.000  IGD 0.0050  Spacing 0.0031


  MODE                   HV 1.195+-0.000  IGD 0.0048  Spacing 0.0045


  MOEA/D-DE              HV 1.193+-0.000  IGD 0.1234  Spacing 0.0827
P10 M=5 (many-objective)


  NSGA-II                HV 0.092+-0.002  IGD 0.680


  NSGA-III               HV 0.484+-0.021  IGD 0.455


  MOEA/D                 HV 0.820+-0.050  IGD 0.438


  RVEA                   HV 0.943+-0.011  IGD 0.326
(HypE is omitted here because one run takes about a minute; see report Task D2.)


In [ ]:
if os.path.exists('outputs/data/taskD.json'):
    D = json.load(open('outputs/data/taskD.json'))
    rk = sorted([(int(k.split('_')[-1]), v) for k, v in D.items() if k.startswith('D1_P3_')], key=lambda r: -r[1]['HV'][0])
    print('Full 10-seed ranking on P3 (from outputs/data/taskD.json):')
    for i, (a, v) in enumerate(rk[:10]): print('%2d %-22s HV %.3f+-%.3f IGD %.4f' % (i + 1, v['name'], *v['HV'], v['IGD'][0]))
else: print('outputs/data not found - unzip the outputs archive next to this notebook to see the 10-seed tables')

Full 10-seed ranking on P3 (from outputs/data/taskD.json):
 1 MOEA/D-DE              HV 0.531+-0.004 IGD 0.0060
 2 MOABC                  HV 0.530+-0.004 IGD 0.0096
 3 PESA-II                HV 0.527+-0.003 IGD 0.0102
 4 MOPSO                  HV 0.522+-0.002 IGD 0.0114
 5 IBEA                   HV 0.521+-0.008 IGD 0.0133
 6 Weight Metric (p-norm, Tchebycheff) HV 0.517+-0.000 IGD 0.0180
 7 eps-Constraint         HV 0.514+-0.000 IGD 0.0389
 8 SMS-EMOA               HV 0.512+-0.010 IGD 0.0169
 9 NSGA-II                HV 0.511+-0.008 IGD 0.0180
10 Constrained NSGA-II    HV 0.511+-0.008 IGD 0.0180


## 6. Algorithms 36–37 on P13 (MGDA and GradNorm)

In [ ]:
from moo.a36_gradient import *
for m in ('equal', 'mgda', 'gradnorm'):
    h = run_toy(m); print('toy %-9s final x = %s, losses = %s, weights = %s' % (m, np.round(h[-1][0], 3), np.round(h[-1][1], 3), np.round(h[-1][2], 2)))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, sc, lr in zip(axs, (1.0, 10.0), (0.05, 0.01)):
    for m, ls in (('equal', '-'), ('mgda', '--'), ('gradnorm', ':')):
        hist, L0 = run_net(m, scale_reg=sc, lr=lr); L = np.array([h[0] for h in hist]) / L0
        ax.plot(L[:, 0], ls, c='C0', label=m + ' cls'); ax.plot(L[:, 1], ls, c='C3', label=m + ' reg'); print('network, regression x%g, %-9s final loss ratios %s' % (sc, m, np.round(L[-1], 3)))
    ax.set_yscale('log'); ax.set_title('loss / initial loss, regression target x%g' % sc); ax.legend(fontsize=6)
plt.show()

toy equal     final x = [0. 0.], losses = [18. 18.], weights = [1. 1.]
toy mgda      final x = [2.5 2.5], losses = [ 0.5 60.5], weights = [1. 1.]
toy gradnorm  final x = [1.327 1.327], losses = [ 5.601 37.437], weights = [1.63 0.37]


network, regression x1, equal     final loss ratios [0.127 0.009]


network, regression x1, mgda      final loss ratios [0.163 0.017]


network, regression x1, gradnorm  final loss ratios [0.097 0.048]


network, regression x10, equal     final loss ratios [0.439 0.015]


network, regression x10, mgda      final loss ratios [0.158 0.019]


network, regression x10, gradnorm  final loss ratios [0.246 0.112]


## Notes
* All 37 algorithms are in `moo/` (grouped by family); each class has `initialize / evaluate / iterate / run`; `tests/test_all.py` holds the unit tests.
* The full reproducible pipeline is `experiments/run_task_a.py`, `run_task_c.py`, `run_task_d.py`, `run_extras.py`, `make_figures.py`, `build_report.py`.